# FluxPipeline - Basic Image Generation

This notebook demonstrates the basics of generating images with FluxPipeline.

## What You'll Learn
- Initialize the FluxPipeline
- Generate your first image
- Understand generation parameters
- Save and display results

## Setup

First, let's import the necessary modules and check our environment.

In [ ]:
# Add parent directory to path so we can import flux_pipeline
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

# Standard imports
import torch
from PIL import Image
import matplotlib.pyplot as plt

# FluxPipeline imports
from pipeline import FluxPipeline
from core import SeedProfile
from config import setup_environment, logger
from utils import setup_workspace

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Initialize the Pipeline

Let's set up the environment and initialize the FluxPipeline.

In [ ]:
# Setup environment
setup_environment()

# Create workspace
workspace = setup_workspace()
print(f"Workspace: {workspace}")

# Initialize pipeline
pipeline = FluxPipeline(workspace=workspace)
print("Pipeline initialized!")

## Load the Model

This step downloads and loads the FLUX.1-schnell model. This may take a few minutes on first run.

In [ ]:
# Load the model
if pipeline.load_model():
    print("✅ Model loaded successfully!")
else:
    print("❌ Failed to load model")
    raise RuntimeError("Model loading failed")

## Generate Your First Image

Now let's generate an image! We'll start with a simple prompt.

In [ ]:
# Define the prompt
prompt = "A serene mountain landscape at sunset, with snow-capped peaks reflecting in a crystal-clear lake"

# Generate image
image, seed = pipeline.generate_image(
    prompt=prompt,
    num_inference_steps=4,  # FLUX.1-schnell works best with 1-4 steps
    guidance_scale=0.0,     # Recommended for FLUX.1-schnell
    height=1024,
    width=1024,
    seed_profile=SeedProfile.BALANCED
)

print(f"Generated with seed: {seed}")

## Display the Result

In [ ]:
if image:
    # Display the image
    plt.figure(figsize=(12, 12))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Generated Image (seed: {seed})")
    plt.tight_layout()
    plt.show()
    
    # Save the image
    output_path = workspace / f"example_basic_{seed}.png"
    image.save(output_path)
    print(f"💾 Saved to: {output_path}")
else:
    print("❌ Image generation failed")

## Understanding Parameters

Let's explore how different parameters affect generation.

In [ ]:
# Try different prompts
prompts = [
    "A futuristic cityscape at night with neon lights",
    "A magical forest with glowing mushrooms and fairy lights",
    "An ancient dragon perched on a crystal castle tower"
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, prompt_text in enumerate(prompts):
    img, seed_used = pipeline.generate_image(
        prompt=prompt_text,
        num_inference_steps=4,
        height=512,  # Smaller for faster generation
        width=512,
        seed_profile=SeedProfile.BALANCED
    )
    
    if img:
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f"Seed: {seed_used}", fontsize=10)

plt.tight_layout()
plt.show()

## Reproducible Generation

Use a specific seed to get reproducible results.

In [ ]:
# Generate with a fixed seed
fixed_seed = 42
prompt = "A beautiful sunset over the ocean"

# Generate twice with the same seed
image1, _ = pipeline.generate_image(
    prompt=prompt,
    seed=fixed_seed,
    num_inference_steps=4,
    height=512,
    width=512
)

image2, _ = pipeline.generate_image(
    prompt=prompt,
    seed=fixed_seed,
    num_inference_steps=4,
    height=512,
    width=512
)

# Display side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.imshow(image1)
ax1.set_title("Generation 1")
ax1.axis('off')
ax2.imshow(image2)
ax2.set_title("Generation 2")
ax2.axis('off')
plt.suptitle(f"Same seed ({fixed_seed}) produces identical results")
plt.show()

print("Images should be identical!")

## Next Steps

Now that you've mastered the basics, try:

1. **[02_batch_processing.ipynb](02_batch_processing.ipynb)** - Generate multiple images at once
2. **[03_gif_creation.ipynb](03_gif_creation.ipynb)** - Create animated GIFs
3. **[04_custom_prompts.ipynb](04_custom_prompts.ipynb)** - Advanced prompt engineering

## Cleanup

Free up GPU memory when done.

In [ ]:
# Clean up
import gc

del pipeline
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
print("✅ Cleanup complete!")